[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E7%BE%A4%E7%B5%84%E8%B3%87%E6%96%99%E5%92%8C%E7%BE%A4%E7%B5%84%E9%81%8B%E7%AE%97/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E5%8F%B0%E5%8D%97%E4%BA%BA%E7%9C%9F%E7%9A%84%E6%84%9B%E5%85%A8%E7%B3%96%E5%97%8E.ipynb)

# 🧩 茶語時光：台南人真的愛全糖嗎？

> 🧋 **情境**：公司內部流傳很多「都市傳說」：台南人特別愛全糖、逢甲店晚上才是戰場、會員比較會花錢……新來的行銷總監說：「別再用猜的，拿數據來！」你手上有第一季 12 萬筆訂單明細，只要學會一招 **groupby（分組 → 計算 → 合併）**，就能一一驗證。

| 素材檔案 | 內容 |
| --- | --- |
| `訂單明細_2024Q1.csv` | 第一季約 12 萬筆訂單明細：時間、門市、品項、甜度、冰塊、數量、金額、是否會員、付款方式 |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '訂單明細_2024Q1.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E7%BE%A4%E7%B5%84%E8%B3%87%E6%96%99%E5%92%8C%E7%BE%A4%E7%B5%84%E9%81%8B%E7%AE%97/%E7%BE%A4%E7%B5%84%E8%B3%87%E6%96%99%E5%92%8C%E7%BE%A4%E7%B5%84%E9%81%8B%E7%AE%97_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['訂單明細_2024Q1.csv']


## 任務 1：各店成績單——營收、杯數、訂單數、客單價

**白話說明**：「依 ___ 分組，計算 ___ 的 ___」。注意：一筆訂單有好幾列（好幾個品項），**訂單數要用 `nunique` 數不重複的訂單編號**。

In [2]:
import pandas as pd

orders = pd.read_csv('訂單明細_2024Q1.csv', parse_dates=['訂單時間'])
print(orders.shape)
print(orders.head(3))

kpi = orders.groupby('門市').agg(營收=('金額', 'sum'),            # (新欄名, (原欄位, 計算方式))
                                 杯數=('數量', 'sum'),
                                 訂單數=('訂單編號', 'nunique'))   # 不重複的訂單編號數
kpi['客單價'] = (kpi['營收'] / kpi['訂單數']).round(1)            # 先加總、再相除
print(kpi.sort_values('營收', ascending=False))

(119948, 13)
            訂單編號                訂單時間     門市     品項  類別 尺寸  甜度  冰塊  數量  單價  金額  \
0  T240101000001 2024-01-01 18:36:29  台北信義店   珍珠奶茶  奶茶  中  微糖  微冰   1  55  55   
1  T240101000001 2024-01-01 18:36:29  台北信義店    多多綠  果茶  中  少糖  少冰   1  50  50   
2  T240101000001 2024-01-01 18:36:29  台北信義店  四季春青茶  純茶  中  半糖  少冰   1  35  35   

  會員  付款方式  
0  N  街口支付  
1  N  街口支付  
2  N  街口支付  
            營收     杯數    訂單數    客單價
門市                                 
台中逢甲店  2172560  37577  17430  124.6
高雄巨蛋店  1761655  31787  15508  113.6
台北西門店  1751735  31381  15350  114.1
台北信義店  1532955  27937  12048  127.2
台南成大店  1521110  27017  14710  103.4


## 任務 2：傳說 1——台南人真的愛全糖嗎？

**白話說明**：依「門市 × 甜度」分組加總杯數，再把每家店換算成百分比——**一定要比「比例」**，因為每家店的總杯數不一樣。

In [3]:
sweet = orders.groupby(['門市', '甜度'])['數量'].sum().unstack('甜度')   # 門市 × 甜度 的杯數
sweet = sweet[['正常糖', '少糖', '半糖', '微糖', '無糖']]
sweet_pct = sweet.div(sweet.sum(axis=1), axis=0) * 100               # 每一列除以該店總杯數 → 百分比
print(sweet_pct.round(1))

甜度      正常糖    少糖    半糖    微糖    無糖
門市                                 
台中逢甲店  22.1  21.5  26.0  16.2  14.1
台北信義店  12.0  17.9  27.5  20.2  22.4
台北西門店  22.2  22.4  25.7  15.7  13.9
台南成大店  41.7  20.1  18.0  12.2   7.9
高雄巨蛋店  21.6  22.3  26.0  15.8  14.3


> 💡 **看懂結果**：台南成大店點「正常糖」的比例是其他門市的 **2～3 倍**——傳說是真的！台北信義店（商辦區）則是「無糖」比例最高。
> 行銷建議：台南主打招牌甜飲，信義店可以優先試賣健康無糖新品。
>
> 🤖 **換你問 AI**：「請依門市與甜度分組加總杯數，換算成每家門市的百分比，並找出各門市最受歡迎的甜度。」

## 任務 3：傳說 2——逢甲店晚上才是戰場？找出各店尖峰時段

In [4]:
orders['小時'] = orders['訂單時間'].dt.hour
by_hour = orders.groupby(['門市', '小時'])['訂單編號'].nunique().unstack('門市')   # 小時 × 門市 的訂單數
print(by_hour)
print('\n各店尖峰時段：')
print(by_hour.idxmax().map(lambda h: f'{h}:00～{h + 1}:00'))       # idxmax：每一欄最大值出現在哪一個小時

門市  台中逢甲店  台北信義店  台北西門店  台南成大店  高雄巨蛋店
小時                                   
10    236    377    408    427    400
11    461    952    679    908    592
12    683   2217   1029   1460    985
13    887   1869   1248   1351    957
14    846   1327   1457   1084    927
15   1076   1474   1660   1387   1176
16   1293   1325   1643   1528   1426
17   1817    957   1610   1324   1343
18   2122    586   1628   1347   1636
19   2383    362   1500   1286   1828
20   2426    185   1262   1053   1789
21   1879    211    795    888   1417
22   1321    206    431    667   1032

各店尖峰時段：
門市
台中逢甲店    20:00～21:00
台北信義店    12:00～13:00
台北西門店    15:00～16:00
台南成大店    16:00～17:00
高雄巨蛋店    19:00～20:00
dtype: object


> 💡 **看懂結果**：信義店的尖峰在中午 12 點（上班族午休），逢甲店的尖峰在晚上 8 點（夜市人潮）——**同一個品牌，排班表應該完全不同**。

## 任務 4：傳說 3——會員比較會花錢嗎？

**白話說明**：明細是「一列一個品項」，要比較「每筆訂單花多少」，必須**先把同一筆訂單加總**（第一次 groupby），再依會員分組比較（第二次 groupby）。

In [5]:
per_order = orders.groupby(['訂單編號', '會員', '門市'], as_index=False)['金額'].sum()   # 第一次：每筆訂單的總金額
member = per_order.groupby('會員')['金額'].agg([('訂單數', 'count'), ('平均客單價', 'mean'),
                                              ('中位數', 'median'), ('最高', 'max')])     # 第二次：會員 vs. 非會員
print(member.round(1))

      訂單數  平均客單價    中位數   最高
會員                          
N   43461  106.1   85.0  545
Y   31585  130.7  115.0  615


> 💡 **看懂結果**：會員的平均客單價明顯高於非會員（約多 25 元），中位數也比較高——**會員比較會花錢是真的**，推廣會員制度值得投資。

## 任務 5：pd.cut——把訂單分成「小額、中額、大額」

In [6]:
per_order['金額級距'] = pd.cut(per_order['金額'], bins=[0, 80, 160, 10_000],
                             labels=['小額(≤80)', '中額(81-160)', '大額(>160)'])
level = pd.crosstab(per_order['門市'], per_order['金額級距'], normalize='index') * 100   # 各店訂單的級距占比 %
print(level.round(1))

金額級距   小額(≤80)  中額(81-160)  大額(>160)
門市                                  
台中逢甲店     36.2        37.3      26.4
台北信義店     35.0        35.8      29.2
台北西門店     42.4        35.6      22.1
台南成大店     50.4        32.3      17.2
高雄巨蛋店     42.9        35.5      21.6


> 💡 **看懂結果**：信義店（辦公室團購）與逢甲店（朋友揪團逛夜市）的大額訂單比例最高；台南成大店以學生為主，一半是單杯小額訂單。**滿額優惠要針對信義、逢甲設計，台南則適合第二杯折扣。**
>
> 🤖 **換你問 AI**：「請先把訂單明細依訂單編號加總成每筆訂單金額，再用 pd.cut 分成 ≤80、81–160、>160 三級，計算各門市的級距占比。」

## 任務 6：樞紐分析表——每家店的「招牌飲料」

**白話說明**：每家店賣最多的都是珍珠奶茶（它本來就是全公司最暢銷的），這樣看不出特色。改看「**偏好指數**」＝某品項在這家店的占比 ÷ 全公司的占比，大於 1 代表「這家店的客人特別愛」。

In [7]:
pivot = orders.pivot_table(index='品項', columns='門市', values='數量', aggfunc='sum', margins=True, margins_name='合計')
print(pivot.sort_values('合計', ascending=False))

share = orders.groupby(['門市', '品項'])['數量'].sum()
share = share / share.groupby(level='門市').transform('sum')              # transform：每個品項除以「自己門市」的總杯數 → 店內占比
overall = orders.groupby('品項')['數量'].sum() / orders['數量'].sum()     # 全公司占比
lift = (share / overall).unstack('門市')                                 # 偏好指數 = 店內占比 ÷ 全公司占比（> 1 代表比平均更受歡迎）
print(lift.round(2))
print('\n偏好指數最高的品項：\n', lift.idxmax())
print('\n排除季節性飲料後的各店招牌：\n', lift.drop(index=['熱可可', '芒果冰沙']).idxmax())

門市        台中逢甲店    台北信義店    台北西門店    台南成大店    高雄巨蛋店      合計
品項                                                         
合計      37577.0  27937.0  31381.0  27017.0  31787.0  155699
珍珠奶茶     8132.0   6176.0   9460.0   6388.0   7109.0   37265
四季春青茶    5595.0   4620.0   4136.0   3831.0   4724.0   22906
黑糖珍珠鮮奶   7411.0   3674.0   3813.0   3607.0   3923.0   22428
鮮奶綠      4142.0   4887.0   3464.0   3064.0   3587.0   19144
冬瓜檸檬     4092.0   2570.0   3118.0   4281.0   3573.0   17634
檸檬紅茶     4290.0   2602.0   3217.0   2957.0   3502.0   16568
多多綠      3591.0   2092.0   2802.0   2473.0   4611.0   15569
熱可可       190.0   1275.0   1332.0     19.0      NaN    2816
芒果冰沙      134.0     41.0     39.0    397.0    758.0    1369
門市      台中逢甲店  台北信義店  台北西門店  台南成大店  高雄巨蛋店
品項                                       
冬瓜檸檬     0.96   0.81   0.88   1.40   0.99
四季春青茶    1.01   1.12   0.90   0.96   1.01
多多綠      0.96   0.75   0.89   0.92   1.45
檸檬紅茶     1.07   0.88   0.96   1.03   1.04
熱可可      0.28   2.52   2.35 

> 💡 **看懂結果**：
> - 偏好指數最高的竟然是季節飲料：台北比較冷 → 熱可可特別好賣；台南、高雄比較熱 → 芒果冰沙特別好賣。**氣候差異會直接反映在菜單上。**
> - 排除季節品後，每家店的招牌就很清楚了：逢甲是黑糖珍珠鮮奶、信義是鮮奶綠、西門是珍珠奶茶、台南是冬瓜檸檬、高雄是多多綠——可以依門市規劃不同的主打商品。

---

## ✍️ 練習題

1. 各門市的付款方式占比（%）是多少？哪家店最常用現金？
2. 依「月份 × 類別」加總杯數，熱飲在三個月之間有什麼變化？
3. 找出每家門市「大杯」的比例，哪家店的客人最愛點大杯？

## 🔑 參考解答（先自己試試看！）

In [8]:
# 練習 1
one_row = orders.drop_duplicates('訂單編號')                            # 一筆訂單只留一列（付款方式以訂單為單位）
pay = pd.crosstab(one_row['門市'], one_row['付款方式'], normalize='index') * 100
print(pay.round(1))
print('最常用現金：', pay['現金'].idxmax())

# 練習 2
orders['月份'] = orders['訂單時間'].dt.month
print(orders.groupby(['月份', '類別'])['數量'].sum().unstack('類別')['熱飲'])

# 練習 3
large = orders.groupby('門市')['尺寸'].apply(lambda s: (s == '大').mean() * 100)
print(large.round(1).sort_values(ascending=False))

付款方式   Apple Pay  LINE Pay   信用卡   悠遊卡    現金  街口支付
門市                                                
台中逢甲店        8.2      21.7   9.8  12.3  36.1  11.9
台北信義店       16.2      25.8  16.5  17.6  16.3   7.6
台北西門店        9.9      22.4  14.2  16.2  29.3   8.1
台南成大店        4.2      20.3   8.0  10.3  45.1  12.2
高雄巨蛋店        8.2      20.1  11.8  11.8  33.8  14.3
最常用現金： 台南成大店
月份
1    1542
2    1031
3     243
Name: 熱飲, dtype: int64
門市
台中逢甲店    45.0
台南成大店    41.4
高雄巨蛋店    35.0
台北西門店    32.9
台北信義店    25.5
Name: 尺寸, dtype: float64
